# Autoencoder Development

#### 1. Dependencies

This section could include:
* Library imports.
* Constant definitions

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.cuda.empty_cache()

In [ ]:
# Constants
BATCH_SIZE = 32
LEARNING_RATE = 0.01
EPOCHS = 30 
outputs = []
losses = [] 

In [ ]:
#Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



#### 2. Data Loading & Feature Engineering

This section could include:

* Loading of data files
* Data manipulation
* Feature engineering strategies

In [ ]:
# Load the dataset
input1 = np.load("subset_1.npy")
input2 = np.load("subset_2.npy")  
input3 = np.load("subset_3.npy")  

# Concatenate all subsets into one array
full_dataset = np.vstack([input1, input2, input3])  # Shape: (total_images, 101250)


In [ ]:

def preprocess_dataset(dataset):
    # Reshape to (num_images, 150, 225, 3) [HWC format]
    dataset = dataset.reshape(-1, 150, 225, 3)

    # Normalize pixel values to [0,1] if in uint8 format
    if dataset.dtype == np.uint8:
        dataset = dataset.astype(np.float32) / 255.0

    # Convert to PyTorch format (num_images, 3, 150, 225) [CHW format]
    dataset = np.transpose(dataset, (0, 3, 1, 2))  # Move channels to first dimension

    return torch.tensor(dataset, dtype=torch.float32)

In [ ]:
# Process dataset
processed_dataset = preprocess_dataset(full_dataset)

#Split into training & test sets (80/20)
train_size = int(0.8 * len(processed_dataset))
test_size = len(processed_dataset) - train_size
train_dataset, test_dataset = random_split(TensorDataset(processed_dataset), [train_size, test_size])

#Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


#### 3. Model Development

This section could include:

* Model Definitions
* Hyperparameter Tuning
* Training code
* Optimisation

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        

        # CR: 2.87
        #Encoder
        # self.encoder = nn.Sequential(
        #     nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),  
        #     nn.ReLU(),
        #     nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), 
        #     nn.ReLU(),
        #     nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1), 
        #     nn.ReLU()

        # )
        

        # # Decoder
        # self.decoder = nn.Sequential(
        #     nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)), 
        #     nn.ReLU(),
        #     nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=(0, 0)),  
        #     nn.ReLU(),
        #     nn.ConvTranspose2d(32, 3, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),  
        #     nn.Sigmoid()  # Normalize output to [0,1]
        # )
        
        #CR: 5.74
        #Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),  # (3,150,225) -> (16,75,113)
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # (16,75,113) -> (16,38,57)
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1), # (32,19,29) -> (32,10,15)
            nn.ReLU()

        )
        

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 32, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),  # (32,10,15) -> (32,19,29)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=(0, 0)),  # (32,38,57) -> (16,75,113)
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),  # (16,75,113) -> (3,150,225)
            nn.Sigmoid()  # Normalize output to [0,1]
        )

        #CR: 11.48
        # Encoder
        # self.encoder = nn.Sequential(
        #     nn.Conv2d(3, 8, kernel_size=3, stride=2, padding=1),  
        #     nn.ReLU(),
        #     nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1), 
        #     nn.ReLU(),
        #     nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1), 
        #     nn.ReLU()

        # )
        

        # # Decoder
        # self.decoder = nn.Sequential(
        #     nn.ConvTranspose2d(16, 16, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),
        #     nn.ReLU(),
        #     nn.ConvTranspose2d(16,8, kernel_size=3, stride=2, padding=1, output_padding=(0, 0)),  
        #     nn.ReLU(),
        #     nn.ConvTranspose2d(8, 3, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),  
        #     nn.Sigmoid()  # Normalize output to [0,1]
        # )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
    
        return decoded
    

In [ ]:

# Initialize Model, Loss Function & Optimizer
model = Autoencoder().to(DEVICE)


# Validation using MSE Loss function
loss_function = nn.MSELoss()

# Using an Adam Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# Training Loop
for epoch in range(EPOCHS):
    epoch_loss = 0  # Track loss for each epoch

    for image_batch in train_loader:  # Assuming loader gives batches of (images, labels)
        # Reshape images:
        image_batch = image_batch[0]  # Extract tensor from tuple

        image_batch = image_batch.view(image_batch.shape[0], 3, 150, 225)

        # # Move to GPU if available
        image_batch = image_batch.to(DEVICE)

        # Forward Pass: Autoencoder Reconstruction
        reconstructed = model(image_batch)

        # Compute Loss
        loss = loss_function(reconstructed, image_batch)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store loss
        epoch_loss += loss.item()
        

    # Save the last batch output for visualization
    outputs.append((epoch, image_batch, reconstructed))

    #store mean epoch loss
    epoch_loss = epoch_loss/len(train_loader)
    losses.append(epoch_loss)
    # Print Loss for each epoch
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {epoch_loss:.4f}")
    
#save the trained model
#torch.save(model.state_dict(), "autoencoder.pth")


#### 4. Model Evaluation

This section could include:

* Testing of models trained
* Generation of results

In [ ]:

# Testing Loop
model.eval()  # Set the model to evaluation mode
test_loss = 0  # Track total loss
outputs_test = [] 

with torch.no_grad():  # Disable gradient calculations  
    for batch_idx, image_batch in enumerate(test_loader):
        image_batch = image_batch[0]  # Extract tensor from tuple
        image_batch = image_batch.to(DEVICE)

        # Forward pass
        reconstructed = model(image_batch)

        # Compute loss
        loss = loss_function(reconstructed, image_batch)
        test_loss += loss.item()

        # Store outputs for visualization
        outputs_test.append((image_batch, reconstructed))

# Compute average test loss
test_loss /= len(test_loader)

# Print Test Loss
print(f"MSE Test Loss: {test_loss:.4f}")
print(len(outputs_test))


#### 5. Figure Creation

This section could include:

* Creation of figures for the report.
* Creating of tables for the report.

In [ ]:
#Epoch loss curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1 ), losses, marker='o', linestyle='-', color='b', label="Training Loss")

plt.xlabel("Epochs")
plt.ylabel("MSE Loss")
plt.title("MSE Training Loss Over Epochs")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# #Batch sizes graph

# # Load MSE loss data from files
# loss_16 = np.load("16_MSE_Losses.npy")
# loss_32 = np.load("32_MSE_Losses.npy")
# loss_64 = np.load("64_MSE_Losses.npy")

# # Create x-axis (epochs) assuming all have the same length
# epochs = np.arange(1, len(loss_16) + 1)

# # Plot each loss curve
# plt.figure(figsize=(8, 5))
# plt.plot(epochs, loss_16, marker='o', linestyle='-', label="16 Batch Size", color='b')
# plt.plot(epochs, loss_32, marker='s', linestyle='--', label="32 Batch Size", color='r')
# plt.plot(epochs, loss_64, marker='^', linestyle='-.', label="64 Batch Size", color='g')

# # Add labels and title
# plt.xlabel("Epochs")
# plt.ylabel("MSE Loss")
# plt.title("MSE Loss Comparison for Different Batch Sizes")
# plt.legend()
# plt.grid(True)

# # Show the plot
# plt.show()

In [ ]:
#Learning rates graph

# # Load MSE loss data from files
# loss_01 = np.load("0.1_MSE_Losses.npy")
# loss_001 = np.load("0.01_MSE_Losses.npy")
# loss_0001= np.load("0.001_MSE_Losses.npy")

# # Create x-axis (epochs) assuming all have the same length
# epochs = np.arange(1, len(loss_01) + 1)

# # Plot each loss curve
# plt.figure(figsize=(8, 5))
# plt.plot(epochs, loss_01, marker='o', linestyle='-', label="0.1 Learning rate", color='b')
# plt.plot(epochs, loss_001, marker='s', linestyle='--', label="0.01 Learning rate", color='r')
# plt.plot(epochs, loss_0001, marker='^', linestyle='-.', label="0.001 Learning rate", color='g')

# # Add labels and title
# plt.xlabel("Epochs")
# plt.ylabel("MSE Loss")
# plt.title("MSE Loss Comparison for Different Learning rates")
# plt.legend()
# plt.grid(True)

# # Show the plot
# plt.show()

In [ ]:
def plot_reconstructed_images(epoch, original_images, reconstructed_images, train = True,  output_dir='figures'):

    # Convert tensors to numpy (HWC format)
    original_images = original_images.detach().cpu().numpy().transpose(0, 2, 3, 1)
    reconstructed_images = reconstructed_images.detach().cpu().numpy().transpose(0, 2, 3, 1)
    
    # Create the figure
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    if train == True:
        O_text = f'Epoch {epoch+1} - Original Image'
        R_text = f'Epoch {epoch+1} - Reconstructed Image'
    else:
        O_text = 'Original Image'
        R_text = 'Reconstructed Image'
    # Plot original image
    axes[0].imshow(original_images[0])  # Plot first image in batch
    axes[0].set_title(O_text)
    axes[0].axis('off')

    # Plot reconstructed image
    axes[1].imshow(reconstructed_images[0])  # Plot first reconstructed image in batch
    axes[1].set_title(R_text)
    axes[1].axis('off')

    # Save figure
    plt.tight_layout()
    
    plt.show()

In [ ]:

# for epoch, original, reconstructed in outputs:
#     plot_reconstructed_images(epoch, original, reconstructed)


for original, reconstructed in outputs_test:
    plot_reconstructed_images(0,original, reconstructed, False)

